# Model 1 - Course to Category classifier

**Task:** supervised multi-class text classification. Input = `Title + Short Intro`,
output = one of 11 categories. Trained on the **frozen split** in `data/processed/`.

Follows `docs/MODEL_PLAN.md`: majority baseline -> TF-IDF + Logistic Regression ->
TF-IDF -> SVD -> MLP. We tune on **dev**, report on **test once**, and write every
final number to `data/processed/` so the report is built from files, not this notebook.

> Run top to bottom. Requires: pandas, numpy, scikit-learn, matplotlib (install via Anaconda).

In [ ]:
import sys, time, json
from pathlib import Path
import numpy as np
import pandas as pd

# Locate the repo root (folder that contains data/processed) so this notebook
# works whether launched from the repo root or from notebooks/.
here = Path.cwd()
REPO_ROOT = here if (here / "data" / "processed").exists() else here.parent
sys.path.insert(0, str(REPO_ROOT))

from src.data_contract import DATA_PROCESSED, LABEL_COL, TEXT_COLS, SEED
np.random.seed(SEED)
print("repo root:", REPO_ROOT)

## 1. Load the frozen split

In [ ]:
def load(name):
    df = pd.read_csv(DATA_PROCESSED / f"{name}.csv")
    # Model 1 input is Title + Short Intro; fill the few missing intros.
    df["text"] = (df["Title"].fillna("") + ". " + df["Short Intro"].fillna("")).str.strip()
    return df

train, dev, test = load("train"), load("dev"), load("test")
print(f"train {len(train)} | dev {len(dev)} | test {len(test)}")
print("classes:", train[LABEL_COL].nunique())
train[[*TEXT_COLS, LABEL_COL, "text"]].head()

In [ ]:
X_train, y_train = train["text"], train[LABEL_COL]
X_dev,   y_dev   = dev["text"],   dev[LABEL_COL]
X_test,  y_test  = test["text"],  test[LABEL_COL]

# Class balance - this is why macro-F1 matters more than accuracy.
train[LABEL_COL].value_counts()

## 2. Metrics helper

Always report **accuracy AND macro-F1** side by side (MODEL_PLAN.md section 6).
Macro-F1 averages over classes so the Business majority cannot flatter the score.

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

def evaluate(name, model, X, y_true, results):
    t0 = time.perf_counter()
    y_pred = model.predict(X)
    infer_s = time.perf_counter() - t0
    acc = accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average="macro")
    print(f"{name:32s}  acc={acc:.3f}  macro-F1={macro_f1:.3f}  "
          f"(infer {infer_s*1000/len(y_true):.2f} ms/item)")
    results.append({"model": name, "accuracy": round(acc, 4),
                    "macro_f1": round(macro_f1, 4),
                    "infer_ms_per_item": round(infer_s*1000/len(y_true), 3)})
    return y_pred

dev_results = []

## 3. Baseline 1 - majority class (the floor)

In [ ]:
from sklearn.dummy import DummyClassifier

majority = DummyClassifier(strategy="most_frequent", random_state=SEED)
majority.fit(X_train, y_train)
evaluate("majority-baseline", majority, X_dev, y_dev, dev_results);

## 4. Baseline 2 - TF-IDF + Logistic Regression

The linear baseline. class_weight='balanced' counteracts the imbalance.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

logreg = Pipeline([
    ("tfidf", TfidfVectorizer(sublinear_tf=True, min_df=2, ngram_range=(1, 2),
                              stop_words="english")),
    ("clf", LogisticRegression(max_iter=1000, class_weight="balanced",
                               random_state=SEED)),
])
t0 = time.perf_counter()
logreg.fit(X_train, y_train)
print(f"trained in {time.perf_counter()-t0:.2f}s")
evaluate("tfidf-logreg", logreg, X_dev, y_dev, dev_results);

## 5. The trained model - TF-IDF -> SVD -> MLP

A real trained neural model: TF-IDF, reduced with truncated SVD, into a small MLP.
Trains in seconds. This is the "not too simple, not too complex" target (section 3).

In [ ]:
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier

mlp = Pipeline([
    ("tfidf", TfidfVectorizer(sublinear_tf=True, min_df=2, ngram_range=(1, 2),
                              stop_words="english")),
    ("svd", TruncatedSVD(n_components=300, random_state=SEED)),
    ("scale", StandardScaler()),
    ("clf", MLPClassifier(hidden_layer_sizes=(256,), max_iter=200,
                          early_stopping=True, random_state=SEED)),
])
t0 = time.perf_counter()
mlp.fit(X_train, y_train)
train_s = time.perf_counter() - t0
print(f"trained in {train_s:.2f}s | epochs run: {mlp.named_steps['clf'].n_iter_}")
evaluate("tfidf-svd-mlp", mlp, X_dev, y_dev, dev_results);

In [ ]:
# Dev comparison table - this IS the 'training statistics' the rubric asks for.
dev_table = pd.DataFrame(dev_results).sort_values("macro_f1", ascending=False)
dev_table

## 6. Final evaluation on TEST - once only

Pick the best model by **dev** macro-F1, then score **test** a single time (section 6:
"dev selects, test reports"). If macro-F1 > 0.90 here, suspect leakage before celebrating.

In [ ]:
best_name = dev_table.iloc[0]["model"]
best_model = {"majority-baseline": majority, "tfidf-logreg": logreg,
              "tfidf-svd-mlp": mlp}[best_name]
print("Best on dev:", best_name)

test_results = []
y_pred = evaluate(f"{best_name} (TEST)", best_model, X_test, y_test, test_results)

print("\nPer-class report (TEST):\n")
print(classification_report(y_test, y_pred, zero_division=0))

In [ ]:
import matplotlib.pyplot as plt

labels = sorted(y_test.unique())
cm = confusion_matrix(y_test, y_pred, labels=labels)
fig, ax = plt.subplots(figsize=(8, 7))
ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels, rotation=90)
ax.set_yticks(range(len(labels))); ax.set_yticklabels(labels)
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
ax.set_title(f"Confusion matrix - {best_name} (test)")
for i in range(len(labels)):
    for j in range(len(labels)):
        ax.text(j, i, cm[i, j], ha="center", va="center",
                color="white" if cm[i, j] > cm.max()/2 else "black", fontsize=8)
fig.tight_layout()
fig.savefig(DATA_PROCESSED / "model1_confusion.png", dpi=120)
plt.show()

## 7. Write results to files

Every number the report cites comes from these files, not the notebook (section 5).

In [ ]:
report = classification_report(y_test, y_pred, zero_division=0, output_dict=True)
out = {
    "seed": SEED,
    "split_sizes": {"train": len(train), "dev": len(dev), "test": len(test)},
    "dev_comparison": dev_results,
    "test_best_model": best_name,
    "test_metrics": test_results[0],
    "test_per_class": {k: v for k, v in report.items()
                       if k not in ("accuracy", "macro avg", "weighted avg")},
    "test_macro_avg": report["macro avg"],
}
(DATA_PROCESSED / "model1_results.json").write_text(
    json.dumps(out, indent=2, ensure_ascii=False), encoding="utf-8")

lines = ["# Model 1 results\n", "_Generated by notebooks/model1_classifier.ipynb._\n",
         "## Dev comparison\n", "| Model | Accuracy | Macro-F1 |", "|---|---|---|"]
for r in dev_results:
    lines.append(f"| {r['model']} | {r['accuracy']:.3f} | {r['macro_f1']:.3f} |")
t = test_results[0]
lines += ["", f"## Test (best = {best_name})\n",
          f"- Accuracy: **{t['accuracy']:.3f}**  Macro-F1: **{t['macro_f1']:.3f}**",
          f"- Inference: {t['infer_ms_per_item']:.2f} ms/item"]
(DATA_PROCESSED / "model1_results.md").write_text("\n".join(lines), encoding="utf-8")
print("Wrote model1_results.json / .md / confusion.png to", DATA_PROCESSED)